In [1]:
%load_ext autoreload
%autoreload 2

In [22]:
import time

import numpy as np
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.termination.collection import TerminationCollection

from sudoku.operators import (
    EPLSurvival,
    LocalSearchRepair,
    MySampling,
    RowCrossover,
    SwapReinitMutation,
    ZeroFunctionValueTermination,
)
from sudoku.problem import SudokuProblem

In [ ]:
easy_no_1 = np.array(
    [
        [0, 0, 9, 0, 0, 0, 1, 0, 0],
        [2, 1, 7, 0, 0, 0, 3, 6, 8],
        [0, 0, 0, 2, 0, 7, 0, 0, 0],
        [0, 6, 4, 1, 0, 3, 5, 8, 0],
        [0, 7, 0, 0, 0, 0, 0, 3, 0],
        [1, 5, 0, 4, 2, 8, 0, 7, 9],
        [0, 0, 0, 5, 8, 9, 0, 0, 0],
        [4, 8, 5, 0, 0, 0, 2, 9, 3],
        [0, 0, 6, 3, 0, 2, 8, 0, 0],
    ]
)

problem = SudokuProblem(initial_board=easy_no_1)

algorithm = GA(
    pop_size=150,
    sampling=MySampling(),
    crossover=RowCrossover(row_cross_rate=0.1, prob=0.2),
    mutation=SwapReinitMutation(swap_rate=0.3, reinit_rate=0.05),
    repair=LocalSearchRepair(),
    survival=EPLSurvival(n_elite=50, sampling=MySampling()),
    eliminate_duplicates=True,
)

# Wang et al. limit to 10,000 gens for fair comparison
termination = TerminationCollection(
    get_termination("n_gen", 10000),
    # get_termination('fmin', 1e-6),
    # fmin = 0 fails because of some weird progress thing
    # 1e-6 is okay, but you get a warning
    # RuntimeWarning: divide by zero encountered in scalar divide
    #   return self.fmin / opt.get("F").min()
    # pymoo's MinimumFunctionValueTermination can't handle 0,
    # so let's use make our own
    ZeroFunctionValueTermination(),
)

start_time = time.perf_counter()
print(start_time)
res = minimize(problem, algorithm, termination, seed=42, verbose=True)
end_time = time.perf_counter()

5261.189235139
n_gen  |  n_eval  |     f_avg     |     f_min    
     1 |      150 |  3.066000E+01 |  1.600000E+01
     2 |      300 |  3.136000E+01 |  1.000000E+01
     3 |      450 |  3.006667E+01 |  1.000000E+01
     4 |      600 |  2.734000E+01 |  8.0000000000
     5 |      750 |  2.308667E+01 |  7.0000000000
     6 |      900 |  2.250667E+01 |  4.0000000000
     7 |     1050 |  2.165333E+01 |  2.0000000000
     8 |     1200 |  2.071333E+01 |  2.0000000000
     9 |     1350 |  1.572667E+01 |  0.000000E+00


In [25]:
res.X.reshape(9, 9)

array([[5, 4, 9, 8, 3, 6, 1, 2, 7],
       [2, 1, 7, 9, 5, 4, 3, 6, 8],
       [6, 3, 8, 2, 1, 7, 9, 5, 4],
       [9, 6, 4, 1, 7, 3, 5, 8, 2],
       [8, 7, 2, 6, 9, 5, 4, 3, 1],
       [1, 5, 3, 4, 2, 8, 6, 7, 9],
       [3, 2, 1, 5, 8, 9, 7, 4, 6],
       [4, 8, 5, 7, 6, 1, 2, 9, 3],
       [7, 9, 6, 3, 4, 2, 8, 1, 5]], dtype=int32)

In [28]:
for i in res.pop[::30]:
    print(i.F)
    print(i.X.reshape(9, 9))

[0.]
[[5 4 9 8 3 6 1 2 7]
 [2 1 7 9 5 4 3 6 8]
 [6 3 8 2 1 7 9 5 4]
 [9 6 4 1 7 3 5 8 2]
 [8 7 2 6 9 5 4 3 1]
 [1 5 3 4 2 8 6 7 9]
 [3 2 1 5 8 9 7 4 6]
 [4 8 5 7 6 1 2 9 3]
 [7 9 6 3 4 2 8 1 5]]
[3.]
[[5 4 9 6 3 8 1 2 7]
 [2 1 7 9 4 5 3 6 8]
 [6 3 8 2 1 7 9 5 4]
 [9 6 4 1 7 3 5 8 2]
 [8 7 2 9 5 6 4 3 1]
 [1 5 3 4 2 8 6 7 9]
 [3 2 1 5 8 9 7 4 6]
 [4 8 5 7 6 1 2 9 3]
 [7 9 6 3 4 2 8 1 5]]
[41.]
[[8 7 9 5 6 4 1 3 2]
 [2 1 7 5 9 4 3 6 8]
 [8 9 4 2 3 7 6 5 1]
 [9 6 4 1 7 3 5 8 2]
 [6 7 4 8 9 2 5 3 1]
 [1 5 3 4 2 8 6 7 9]
 [7 6 2 5 8 9 1 3 4]
 [4 8 5 7 1 6 2 9 3]
 [4 5 6 3 7 2 8 1 9]]
[3.]
[[5 4 9 6 3 8 1 2 7]
 [2 1 7 9 5 4 3 6 8]
 [6 3 8 2 1 7 9 5 4]
 [9 6 4 1 7 3 5 8 2]
 [8 7 2 9 6 5 4 3 1]
 [1 5 3 4 2 8 6 7 9]
 [3 2 1 5 8 9 7 4 6]
 [4 8 5 7 1 6 2 9 3]
 [7 9 6 3 4 2 8 1 5]]
[47.]
[[2 3 9 4 5 7 1 6 8]
 [2 1 7 4 5 9 3 6 8]
 [3 1 4 2 6 7 8 5 9]
 [9 6 4 1 7 3 5 8 2]
 [6 7 8 1 2 9 4 3 5]
 [1 5 6 4 2 8 3 7 9]
 [7 6 1 5 8 9 2 4 3]
 [4 8 5 6 7 1 2 9 3]
 [4 7 6 3 5 2 8 1 9]]
